# Multi-document reasoning: the real-library code of the lessons

Booster LEAP, Advanced RAG Engineering. The cells below are the library versions of the examples in the lessons (the lessons first build the same mechanism in plain Python in your browser). Each cell was run once and its output is saved below it.

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use it:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). Change one thing at a time and run again.

**What needs what:** the first run downloads the small embedding model all-MiniLM-L6-v2 (about 90 MB; it needs internet but no GPU and no key). No secret or key is used in this notebook. The install line pins the versions that were used to produce the saved outputs; Colab may resolve the dependencies of those packages differently.

In [ ]:
%pip -q install sentence-transformers==6.3.0 numpy==2.5.2 langchain-core==1.8.4 pandas==2.3.3

## Context budgeting: what fits in the window

all-MiniLM-L6-v2: the same ranking as the saved one, and words against tokens for six records.

In [ ]:
docs = {
    "help-trash": "Deleted tickets stay in the trash for 30 days and are then removed permanently.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "notes-trash-old": "Version 3.8: deleted tickets are kept in the trash for 14 days before removal.",
    "faq-backup": "Backups still hold removed tickets until they are overwritten after a further 35 days.",
    "mail-trash": "Reminder: deleted tickets stay in the trash for 30 days and then they are removed for good.",
    "chat-trash": "I think the trash keeps deleted tickets for 60 days, someone in support told me.",
}

import os
os.environ["HF_HUB_OFFLINE"] = "1"          # use the model already downloaded
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")
question = "how long can a deleted ticket be recovered, including backups"
scores = util.cos_sim(model.encode(question), model.encode(list(docs.values())))[0]
ranked = [list(docs)[i] for i in scores.argsort(descending=True).tolist()]
print("ranked:", ranked)
for rid in ranked:
    tokens = len(model.tokenizer(docs[rid], add_special_tokens=False)["input_ids"])
    print(f"{rid:16s} words {len(docs[rid].split()):2d}  tokens {tokens}")

ranked: ['help-trash', 'faq-trash', 'notes-trash-old', 'faq-backup', 'mail-trash', 'chat-trash']
help-trash       words 14  tokens 15
faq-trash        words 15  tokens 16
notes-trash-old  words 14  tokens 18
faq-backup       words 14  tokens 17
mail-trash       words 17  tokens 19
chat-trash       words 15  tokens 17


## Stuff, map-reduce and refine

langchain-core: batch for the map step, with a plain function in the place of the model.

In [ ]:
docs = {
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-export": "Admins can export all tickets as CSV from Settings, then Data. The export link works for 24 hours.",
    "faq-backup": "Backups still hold removed tickets until they are overwritten after a further 35 days.",
    "faq-cancel": "Owners can cancel in Settings, then Billing. Service continues until the end of the paid month.",
}

from langchain_core.runnables import RunnableLambda

def map_model(item):
    """a plain function in the place of the model"""
    cid, text = item
    hit = "trash" in text.lower() or "backup" in text.lower()
    return {"id": cid, "note": text if hit else "NOTHING"}

mapper = RunnableLambda(map_model)
notes = mapper.batch(list(docs.items()), config={"max_concurrency": 4})
print("order kept:", [n["id"] for n in notes])
print("with an answer:", [n["id"] for n in notes if n["note"] != "NOTHING"])

reducer = RunnableLambda(lambda ns: " ".join(f"{n['note']} [{n['id']}]" for n in ns if n["note"] != "NOTHING"))
print(reducer.invoke(notes))

order kept: ['faq-trash', 'faq-export', 'faq-backup', 'faq-cancel']
with an answer: ['faq-trash', 'faq-backup']
Deleted tickets stay in the trash for 30 days and are then removed for good. [faq-trash] Backups still hold removed tickets until they are overwritten after a further 35 days. [faq-backup]


## Duplicate and contradictory sources

all-MiniLM-L6-v2: similarity of the other records to faq-trash, computed live.

In [ ]:
docs = {
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "help-trash": "Deleted tickets stay in the trash for 30 days and are then removed permanently.",
    "mail-trash": "Reminder: deleted tickets stay in the trash for 30 days and then they are removed for good.",
    "notes-trash-old": "Version 3.8: deleted tickets are kept in the trash for 14 days before removal.",
    "chat-trash": "I think the trash keeps deleted tickets for 60 days, someone in support told me.",
    "faq-backup": "Backups still hold removed tickets until they are overwritten after a further 35 days.",
}

import os
os.environ["HF_HUB_OFFLINE"] = "1"          # use the model already downloaded
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")
vectors = model.encode(list(docs.values()), normalize_embeddings=True)
scores = util.cos_sim(vectors[0], vectors)[0].tolist()
for rid, s in sorted(zip(docs, scores), key=lambda p: -p[1])[1:]:
    print(f"{rid:16s} similarity {s:.2f}")

help-trash       similarity 0.99
mail-trash       similarity 0.93
notes-trash-old  similarity 0.88
chat-trash       similarity 0.84
faq-backup       similarity 0.73


## Multi-hop retrieval

all-MiniLM-L6-v2: one search against two hops.

In [ ]:
docs = {
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "help-trash": "Deleted tickets stay in the trash for 30 days and are then removed permanently.",
    "mail-trash": "Reminder: deleted tickets stay in the trash for 30 days and then they are removed for good.",
    "notes-trash-old": "Version 3.8: deleted tickets are kept in the trash for 14 days before removal.",
    "chat-trash": "I think the trash keeps deleted tickets for 60 days, someone in support told me.",
    "faq-backup": "Backups still hold removed tickets until they are overwritten after a further 35 days.",
    "faq-export": "Admins can export all tickets as CSV from Settings, then Data. The export link works for 24 hours.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-cancel": "Owners can cancel in Settings, then Billing. Service continues until the end of the paid month.",
    "price-starter": "Starter plan: up to 5 seats, 0 per seat per month, no single sign-on, community forum support.",
    "price-team": "Team plan: up to 50 seats, 12 per seat per month, no single sign-on, email support within 2 business days.",
    "price-business": "Business plan: up to 500 seats, 20 per seat per month, single sign-on included, chat support within 4 hours.",
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "svc-export": "The ticket export service is owned by the Data team.",
    "svc-signin": "The sign-in service is owned by the Identity team.",
    "svc-mail": "The mail intake service for customer email is owned by the Messaging team.",
    "svc-billing": "The billing service is owned by the Revenue team.",
    "team-data": "Data team: lead Priya Nair, on-call channel oncall-data.",
    "team-identity": "Identity team: lead Arjun Rao, on-call channel oncall-identity.",
    "team-messaging": "Messaging team: lead Sara Thomas, on-call channel oncall-messaging.",
    "team-revenue": "Revenue team: lead Kabir Shah, on-call channel oncall-revenue.",
}
import re

def next_query(record_text, asked):
    """read the team name from a service record; None if there is nothing to follow"""
    found = re.search(r"owned by the (\w+) team", record_text)
    return f"{found.group(1)} team {asked}" if found else None

from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")
ids = list(docs)
vectors = model.encode([docs[i] for i in ids], normalize_embeddings=True, show_progress_bar=False)

def search(query):
    q = model.encode(query, normalize_embeddings=True, show_progress_bar=False)
    scores = util.cos_sim(q, vectors)[0].tolist()
    return [i for _, i in sorted(zip(scores, ids), key=lambda p: -p[0])]

question = "who leads the team that owns the ticket export service"
print("one search, rank of team-data:", search(question).index("team-data") + 1)
first = search("which team owns the ticket export service")[0]
query = next_query(docs[first], "lead")
second = next(d for d in search(query) if d != first)
print("hop 1:", first)
print("hop 2:", second, "from the query", repr(query))

one search, rank of team-data: 22
hop 1: svc-export
hop 2: team-data from the query 'Data team lead'


## Tabular and structured retrieval

sqlite3: a parameterised query, an average and a hostile value.

In [ ]:
import csv
import io

table = """plan,max_seats,price_per_seat_month,sso,support
Starter,5,0,no,community forum
Team,50,12,no,email within 2 business days
Business,500,20,yes,chat within 4 hours
Enterprise,unlimited,custom,yes,named engineer
"""

import sqlite3

db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE plans (plan TEXT, max_seats INTEGER, price INTEGER, sso TEXT)")
for r in csv.DictReader(io.StringIO(table)):
    seats = 1_000_000_000 if r["max_seats"] == "unlimited" else int(r["max_seats"])   # a large stand-in
    price = None if r["price_per_seat_month"] == "custom" else int(r["price_per_seat_month"])  # unknown = NULL
    db.execute("INSERT INTO plans VALUES (?, ?, ?, ?)", (r["plan"], seats, price, r["sso"]))

sql = "SELECT plan, price * ? FROM plans WHERE sso = ? AND max_seats >= ? ORDER BY price IS NULL, price"
print(db.execute(sql, (40, "yes", 40)).fetchall())
print("average price:", round(db.execute("SELECT AVG(price) FROM plans").fetchone()[0], 2))
print("hostile value:", db.execute("SELECT plan FROM plans WHERE plan = ?", ("x' OR '1'='1",)).fetchall())

[('Business', 800), ('Enterprise', None)]
average price: 10.67
hostile value: []


## Retrieval for agents and code

all-MiniLM-L6-v2: name only against name plus docstring.

In [ ]:
from sentence_transformers import SentenceTransformer, util

functions = {   # name: first line of the docstring
    "purge_trash": "Remove deleted tickets that are older than the retention period.",
    "export_csv": "Write all tickets to a spreadsheet file for an admin to download.",
    "check_sso": "Verify the login certificate used for single sign-on.",
    "bill_total": "Compute the monthly charge from the seat count and the plan price.",
    "route_mail": "Turn an incoming customer email into a new ticket.",
    "rollback": "Return the service to the previous release.",
}
questions = {"how do we permanently erase old items from the bin": "purge_trash",
             "download everything as a spreadsheet": "export_csv",
             "why can staff not log in with the company account": "check_sso",
             "how much does a customer pay each month": "bill_total",
             "a customer wrote to us, how does it become a case": "route_mail",
             "undo a bad deployment": "rollback"}

model = SentenceTransformer("all-MiniLM-L6-v2")
names = list(functions)
versions = {"name only": [n.replace("_", " ") for n in names],
            "name + docstring": [n.replace("_", " ") + ": " + functions[n] for n in names]}

for label, texts in versions.items():
    vectors = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    hits = 0
    for question, right in questions.items():
        q = model.encode(question, normalize_embeddings=True, show_progress_bar=False)
        scores = util.cos_sim(q, vectors)[0].tolist()
        order = [n for _, n in sorted(zip(scores, names), key=lambda p: -p[0])]
        hits += order[0] == right
        if order[0] != right:
            print(f"  {label}: wrong for '{question}': got {order[0]}, right one at rank {order.index(right) + 1}")
    print(f"{label:16s} right function first: {hits} of {len(questions)}")

  name only: wrong for 'a customer wrote to us, how does it become a case': got rollback, right one at rank 4
name only        right function first: 5 of 6
name + docstring right function first: 6 of 6


## Notebook: multi-document reasoning end to end

all-MiniLM-L6-v2: the live top 3 of every search against the saved ranking.

In [ ]:
docs = {
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "help-trash": "Deleted tickets stay in the trash for 30 days and are then removed permanently.",
    "mail-trash": "Reminder: deleted tickets stay in the trash for 30 days and then they are removed for good.",
    "notes-trash-old": "Version 3.8: deleted tickets are kept in the trash for 14 days before removal.",
    "chat-trash": "I think the trash keeps deleted tickets for 60 days, someone in support told me.",
    "faq-backup": "Backups still hold removed tickets until they are overwritten after a further 35 days.",
    "faq-export": "Admins can export all tickets as CSV from Settings, then Data. The export link works for 24 hours.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-cancel": "Owners can cancel in Settings, then Billing. Service continues until the end of the paid month.",
    "price-starter": "Starter plan: up to 5 seats, 0 per seat per month, no single sign-on, community forum support.",
    "price-team": "Team plan: up to 50 seats, 12 per seat per month, no single sign-on, email support within 2 business days.",
    "price-business": "Business plan: up to 500 seats, 20 per seat per month, single sign-on included, chat support within 4 hours.",
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "svc-export": "The ticket export service is owned by the Data team.",
    "svc-signin": "The sign-in service is owned by the Identity team.",
    "svc-mail": "The mail intake service for customer email is owned by the Messaging team.",
    "svc-billing": "The billing service is owned by the Revenue team.",
    "team-data": "Data team: lead Priya Nair, on-call channel oncall-data.",
    "team-identity": "Identity team: lead Arjun Rao, on-call channel oncall-identity.",
    "team-messaging": "Messaging team: lead Sara Thomas, on-call channel oncall-messaging.",
    "team-revenue": "Revenue team: lead Kabir Shah, on-call channel oncall-revenue.",
}
saved = {
"how long do deleted tickets stay in the trash": ["help-trash", "faq-trash", "notes-trash-old"], "how long are backups kept after tickets are removed": ["faq-backup", "help-trash", "faq-trash"], "which plan has single sign-on": ["faq-sso", "price-starter", "svc-signin"], "Business plan price per seat": ["price-business", "price-starter", "faq-uptime"], "Business plan uptime commitment": ["faq-uptime", "price-business", "faq-sso"], "where is customer data stored": ["faq-region", "svc-export", "svc-mail"], "how many leave days can be carried over": ["leave-carry", "leave-annual", "leave-apply"], "how many days before leave must I request it": ["leave-apply", "leave-carry", "leave-annual"], "which team owns the ticket export service": ["svc-export", "faq-export", "svc-billing"], "Data team lead": ["team-data", "svc-export", "team-revenue"], "which team owns the sign-in service": ["svc-signin", "faq-sso", "svc-export"], "Identity team on-call channel": ["team-identity", "team-messaging", "svc-signin"], "which team owns the billing service": ["svc-billing", "faq-cancel", "svc-export"], "Revenue team lead": ["svc-billing", "team-revenue", "svc-export"], "which team owns the mail intake service": ["svc-mail", "svc-billing", "svc-signin"], "Messaging team lead": ["team-messaging", "svc-mail", "team-revenue"], "how long can a deleted ticket be recovered, including backups": ["help-trash", "faq-trash", "notes-trash-old"], "which plan has single sign-on and what does it cost per seat": ["price-starter", "price-business", "faq-sso"], "what uptime is promised on Business and where can its data be stored": ["faq-uptime", "faq-region", "faq-export"], "how many leave days can I carry over and how early must I ask for leave": ["leave-carry", "leave-apply", "leave-annual"], "who leads the team that owns the ticket export service": ["svc-export", "faq-export", "svc-billing"], "which on-call channel handles problems with sign-in": ["team-identity", "team-messaging", "svc-signin"], "who leads the team behind billing": ["svc-billing", "team-revenue", "svc-export"], "who leads the team that looks after incoming customer email": ["svc-mail", "team-messaging", "svc-billing"]
}

from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")
ids = list(docs)
vectors = model.encode([docs[i] for i in ids], normalize_embeddings=True, show_progress_bar=False)

same = 0
for query, expected in saved.items():
    q = model.encode(query, normalize_embeddings=True, show_progress_bar=False)
    scores = util.cos_sim(q, vectors)[0].tolist()
    top = [i for _, i in sorted(zip(scores, ids), key=lambda p: -p[0])][:3]
    same += top == expected
    if top != expected:
        print("differs:", query, top, expected)
print("searches checked:", len(saved), "| same top 3 as saved:", same)
assert same == len(saved)

searches checked: 24 | same top 3 as saved: 24
